## SPARCS Hospital Inpatient Discharges (2024)

Dataset: https://health.data.ny.gov/d/sf4k-39ay

API documentation: https://dev.socrata.com/foundry/health.data.ny.gov/sf4k-39ay

### How the data is collected

The download logic lives in [`src/download.py`](../src/download.py) so it can run as a plain script:

```bash
python -m src.download           # resumes if interrupted
python -m src.download --force   # start over
```

- With over 2 million rows, we request `PAGE_SIZE` rows at a time, moving the `$offset` forward each time. We order by `:id` (a built-in unique row id) so the pages are stable and don't overlap or skip rows.
- Each page is saved as its own parquet file in `data/raw/sparcs_2024/`. If the download stops partway, rerunning it skips pages that are already saved. A `_SUCCESS` marker is written when the last page is in.
- **App token (optional):** if `APP_TOKEN` is set in the `.env` file at the repo root, it is sent as the `X-App-Token` header to avoid throttling.

This notebook calls the same functions, so running it is equivalent to running the script.

### Imports

In [1]:
from src.config import RAW_DIR, PAGE_SIZE
from src.download import download, load_raw, fetch_row_count

### Fetch raw data

Does nothing if the download is already complete.

In [2]:
download()

Raw data already complete in C:\Users\nguye\hospital-stay-forecast\data\raw\sparcs_2024


In [3]:
df = load_raw()
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 2,196,737
Columns: 33


### Sanity check

The local row count should match the API's `count(*)`.

In [4]:
total_rows = fetch_row_count()

if len(df) == total_rows:
    print(f"Row count matches the API: {len(df):,} rows")
else:
    print(f"WARNING: row count mismatch. API has {total_rows:,} rows, DataFrame has {len(df):,}")

Row count matches the API: 2,196,737 rows


In [5]:
df.head()

,health_service_area,hospital_county,operating_certificate_number,permanent_facility_id,facility_name,age_group,zip_code,gender,race,ethnicity,...,apr_severity_of_illness,apr_risk_of_mortality,apr_medical_surgical,payment_typology_1,payment_typology_2,payment_typology_3,birth_weight,emergency_department_indicator,total_charges,total_costs
0,Hudson Valley,Westchester,5957001,001139,WESTCHESTER MEDICAL CENTER,0-17,OOS,F,White,Not Span/Hispanic,...,Moderate,Minor,Medical,Private Health Insurance,<NA>,<NA>,<NA>,Y,46814,6772.07
1,New York City,Queens,7003001,001628,FLUSHING HOSPITAL MEDICAL CENTER,0-17,113,M,White,Spanish/Hispanic,...,Moderate,Moderate,Medical,Medicaid,<NA>,<NA>,<NA>,Y,13490,15464.3
2,New York City,New York,7002054,001458,NEW YORK-PRESBYTERIAN HOSPITAL - NEW YORK WEIL...,70 or Older,100,M,White,Not Span/Hispanic,...,Moderate,Moderate,Medical,Medicare,Private Health Insurance,<NA>,<NA>,Y,49503.16,9324.77
3,New York City,New York,7002054,001464,NEW YORK-PRESBYTERIAN HOSPITAL - COLUMBIA PRES...,0-17,100,F,Other Race,Not Span/Hispanic,...,Minor,Minor,Medical,Private Health Insurance,<NA>,<NA>,2700,Y,27827.66,7304.27
4,New York City,New York,7002032,001466,MOUNT SINAI WEST,18-29,100,F,Other Race,Spanish/Hispanic,...,Moderate,Minor,Medical,Medicare,<NA>,<NA>,<NA>,Y,32798.29,7948.1
